In [1]:
import os

def show_inputs(depth=4, max_files_per_dir=6):
    """Print what is attached under /kaggle/input - check it before trusting the rest."""
    root0 = os.environ.get("SAT_INPUT", "/kaggle/input")
    for root, dirs, files in os.walk(root0):
        dirs[:] = sorted(d for d in dirs if not d.startswith("."))
        level = root.replace(root0, "").count(os.sep)
        if level > depth:
            dirs[:] = []
            continue
        print("  " * level + root + "/")
        for fn in sorted(files)[:max_files_per_dir]:
            print("  " * level + "  " + fn)
        if len(files) > max_files_per_dir:
            print("  " * level + f"  ... +{len(files) - max_files_per_dir} more files")

show_inputs()

/kaggle/input/
  /kaggle/input/datasets/
    /kaggle/input/datasets/rollsroyce1/
      /kaggle/input/datasets/rollsroyce1/sat-ext/
        /kaggle/input/datasets/rollsroyce1/sat-ext/sat_ext/
          README.md
          analysis.py
          audit.py
          config.py
          dataset.py
          dota.py
          ... +10 more files
      /kaggle/input/datasets/rollsroyce1/sat-splits/
        /kaggle/input/datasets/rollsroyce1/sat-splits/audit/
        /kaggle/input/datasets/rollsroyce1/sat-splits/splits/
      /kaggle/input/datasets/rollsroyce1/sat-splits-extra/
        /kaggle/input/datasets/rollsroyce1/sat-splits-extra/sat_splits_extra/
      /kaggle/input/datasets/rollsroyce1/satelite-coco/
        README.dataset.txt
        README.roboflow.txt
        /kaggle/input/datasets/rollsroyce1/satelite-coco/test/
          1011_jpeg.rf.141ad0f26ab02f1bfee50847febecb85.jpg
          1016_jpeg.rf.0aef8e3664a12b1681da06035362ed54.jpg
          1025_jpeg.rf.ce6b95f0170aab49e8aab8728fca22

In [2]:
import os, sys, warnings
warnings.filterwarnings("ignore")
INPUT_ROOT = os.environ.get("SAT_INPUT", "/kaggle/input")

def _find_code_root():
    for root, dirs, files in os.walk(INPUT_ROOT):
        if {"config.py", "audit.py", "metrics.py", "infer.py"} <= set(files):
            return root
    return None

CODE_ROOT = os.environ.get("SAT_CODE") or _find_code_root()
assert CODE_ROOT, "sat_ext code folder not found - Add Input -> your 'sat-ext' code Dataset"
sys.path.insert(0, CODE_ROOT)
print("CODE_ROOT:", CODE_ROOT)

import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 70)
import config as cfg, utils
utils.ensure_dirs(); utils.set_seed(cfg.SEED)
print("work dir:", cfg.WORK_DIR, "| smoke test:", cfg.SMOKE)

CODE_ROOT: /kaggle/input/datasets/rollsroyce1/sat-ext/sat_ext
work dir: /kaggle/working | smoke test: False


In [3]:
KEY = "07K"            # preset - do not change
PILOT = False         # True = 1-epoch YOLO11n check of the whole GPU pipeline (tagged __pilot, ignored by the tables)
import plans, train
PLAN = plans.PILOT if PILOT else plans.PLANS[KEY]
EPOCHS = 1 if (PILOT or cfg.SMOKE) else None
TAG = "pilot" if PILOT else ""
if PILOT:
    print("PILOT: grouped / YOLO11n / seed 42, 1 epoch")
else:
    plans.show(KEY)
print("epochs:", EPOCHS or cfg.TRAIN_PARAMS["epochs"])

07K-train-native-tiles-yolo11m: 3 runs, ~3.0 GPU h (planning estimate)
  split=native_tiles model=YOLO11m    seed=42
  split=native_tiles model=YOLO11m    seed=43
  split=native_tiles model=YOLO11m    seed=44
epochs: 50


In [4]:
utils.require_gpu()
assert utils.install_detection_stack(), "pip install failed - is Internet ON? (Settings -> Internet)"
print(utils.versions())

GPU: {'device': 'cuda:0', 'gpu': 'Tesla T4', 'n_gpu': 2, 'torch': '2.11.0+cu128'}
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
{'ultralytics': '8.4.37', 'sahi': '0.12.8', 'torch': '2.11.0+cu128', 'pycocotools': 'installed', 'numpy': '2.1.3', 'cv2': '4.14.0'}


In [5]:
done = train.run_plan(PLAN, epochs=EPOCHS, tag=TAG)

materialized 'native_tiles' (40ad0ef2ec1e) -> /kaggle/working/data_native_tiles  images {'train': 2992, 'val': 80, 'test': 82}
New https://pypi.org/project/ultralytics/8.4.183 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.37 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_native_tiles/data.yaml, degrees=45.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, l

2026-10-21 13:00:52,384 - sahi - WARNING - ignoring invalid prediction with bbox: [1117.161376953125, 0, 1406.036865234375, 0] (ultralytics.py:278)


  pycocotools vs internal AP (sahi640): {'AP_all': (0.1116110676731935, 0.1116110676731935), 'AP50_all': (0.3824372916827534, 0.3824372916827534), 'AP_small': (0.048035029161174844, 0.048035029161174844)}
  native_tiles__YOLO11m__s42   sahi640   AP50=0.3824 AP=0.1116 AR50_micro=0.597 | conf>=0.25: dets=4447 TP=1659 FP=2788 R=0.589 | 285.3 ms/img


2026-10-21 13:01:46,839 - sahi - WARNING - ignoring invalid prediction with bbox: [1117.161376953125, 0, 1406.036865234375, 0] (ultralytics.py:278)


  pycocotools vs internal AP (sahi480): {'AP_all': (0.11701167716756426, 0.11701167716756426), 'AP50_all': (0.37517535305100536, 0.37517535305100536), 'AP_small': (0.05083651759972004, 0.05083651759972004)}
  native_tiles__YOLO11m__s42   sahi480   AP50=0.3752 AP=0.1170 AR50_micro=0.753 | conf>=0.25: dets=4634 TP=1580 FP=3054 R=0.561 | 492.8 ms/img


...

  native_tiles__YOLO11m__s44   sahi640   AP50=0.7373 AP=0.4744 AR50_micro=0.922 | conf>=0.25: dets=4402 TP=2321 FP=2081 R=0.824 | 282.6 ms/img


2026-10-21 16:36:23,179 - sahi - WARNING - ignoring invalid prediction with bbox: [368.7566833496094, 0, 486.72418212890625, 0] (ultralytics.py:278)
2026-10-21 16:36:23,180 - sahi - WARNING - ignoring invalid prediction with bbox: [359.8095397949219, 0, 458.775146484375, 0] (ultralytics.py:278)


  native_tiles__YOLO11m__s44   sahi480   AP50=0.6884 AP=0.4354 AR50_micro=0.818 | conf>=0.25: dets=4109 TP=2256 FP=1853 R=0.801 | 492.6 ms/img


2026-10-21 16:37:40,320 - sahi - WARNING - ignoring invalid prediction with bbox: [368.7566833496094, 0, 486.72418212890625, 0] (ultralytics.py:278)
2026-10-21 16:37:40,321 - sahi - WARNING - ignoring invalid prediction with bbox: [359.8095397949219, 0, 458.775146484375, 0] (ultralytics.py:278)


  native_tiles__YOLO11m__s44   sahi320   AP50=0.6237 AP=0.3950 AR50_micro=0.844 | conf>=0.25: dets=4207 TP=2101 FP=2106 R=0.746 | 952.3 ms/img

=== session summary ===
  native_tiles__YOLO11m__s42     trained+evaluated
  native_tiles__YOLO11m__s43     trained+evaluated
  native_tiles__YOLO11m__s44     trained+evaluated
session time used: 5.51 h


In [6]:
import shutil, glob
for rid, status in done:
    ev = os.path.join(cfg.RESULTS_DIR, rid, "eval_test.json")
    if os.path.isfile(ev):
        d = utils.load_json(ev)
        u = d["ultralytics_test"]
        print(f"{rid:<30} test mAP50={u.get('mAP50', float('nan')):.4f} mAP50-95={u.get('mAP50_95', float('nan')):.4f} | " +
              " ".join(f"{m}:AP={v['coco'].get('AP_all', float('nan')):.3f}" for m, v in d["modes"].items()))
z = shutil.make_archive(os.path.join(cfg.WORK_DIR, f"results_{KEY}"), "zip", cfg.RESULTS_DIR)
print("\nsmall results zip (no weights):", z, f"{os.path.getsize(z) / 1e6:.1f} MB")
for f in glob.glob(os.path.join(cfg.WORK_DIR, "runs")) + glob.glob(os.path.join(cfg.WORK_DIR, "data_*")):
    shutil.rmtree(f, ignore_errors=True)       # Ultralytics run folders are duplicated in results/ + weights/

native_tiles__YOLO11m__s42     test mAP50=0.5380 mAP50-95=0.2739 | standard:AP=0.041 sahi640:AP=0.112 sahi480:AP=0.117 sahi320:AP=0.140
native_tiles__YOLO11m__s43     test mAP50=0.4620 mAP50-95=0.1587 | standard:AP=0.141 sahi640:AP=0.282 sahi480:AP=0.232 sahi320:AP=0.203
native_tiles__YOLO11m__s44     test mAP50=0.5957 mAP50-95=0.3221 | standard:AP=0.321 sahi640:AP=0.474 sahi480:AP=0.435 sahi320:AP=0.395

small results zip (no weights): /kaggle/working/results_07K.zip 20.4 MB
